# SVTR OCR v3 Unified - Stable Training
نسخة v3 الناجحة مع تحسينات طفيفة على unified_ocr_dataset

In [1]:
import os, math, numpy as np, pandas as pd, cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import models
from tqdm import tqdm
import albumentations as A
from sklearn.model_selection import train_test_split

class Config:
    IMG_HEIGHT, IMG_WIDTH = 64, 400
    BATCH_SIZE = 32
    EPOCHS = 60  # زيادة طفيفة من 50
    LR = 3e-4  # نفس LR الناجح من v3
    D_MODEL = 384  # نفس v3
    NHEAD = 8
    NUM_LAYERS = 6  # نفس v3
    DIM_FF = 1024
    DROPOUT = 0.15
    PATIENCE = 5  # Early stopping patience
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # مسارات unified_ocr_dataset
    BASE_DIR = r"c:\Users\moom\vignette\lasterocr"
    DATASET_DIR = os.path.join(BASE_DIR, "unified_ocr_dataset")
    IMAGES_DIR = os.path.join(DATASET_DIR, "images")
    REAL_CSV = os.path.join(DATASET_DIR, "labels_real.csv")
    SYNTHETIC_CSV = os.path.join(DATASET_DIR, "labels_synthetic.csv")

print(f"Device: {Config.DEVICE}")

/usr/local/lib/python3.11/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()


Device: cpu


In [2]:
# Build vocabulary from both datasets
def build_vocab():
    chars = set()
    
    # Real data
    df_real = pd.read_csv(Config.REAL_CSV)
    for t in df_real['text'].dropna():
        chars.update(str(t))
    
    # Synthetic data
    df_synth = pd.read_csv(Config.SYNTHETIC_CSV)
    for t in df_synth['text'].dropna():
        chars.update(str(t))
    
    return ''.join(sorted(chars))

CHAR_VECTOR = build_vocab()
BLANK_IDX = 0
char_to_idx = {c: i+1 for i, c in enumerate(CHAR_VECTOR)}
idx_to_char = {i+1: c for i, c in enumerate(CHAR_VECTOR)}
NUM_CLASSES = len(CHAR_VECTOR) + 1

print(f"Vocabulary size: {NUM_CLASSES}")
print(f"Characters: {CHAR_VECTOR}")

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\moom\\vignette\\lasterocr/unified_ocr_dataset/labels_real.csv'

In [ ]:
# Dataset with mild augmentation (same as v3)
class OCRDataset(Dataset):
    def __init__(self, paths, labels, augment=False):
        self.paths = paths
        self.labels = labels
        self.augment = augment
        
        # Mild augmentation like v3
        self.aug = A.Compose([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.GaussianBlur(blur_limit=(3, 5), p=0.2),
            A.Affine(scale=(0.95, 1.05), rotate=(-2, 2), p=0.3),
            A.GaussNoise(var_limit=(10, 50), p=0.2),
        ])
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx])
        if img is None:
            img = np.zeros((Config.IMG_HEIGHT, Config.IMG_WIDTH, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if self.augment:
            img = self.aug(image=img)['image']
        
        img = cv2.resize(img, (Config.IMG_WIDTH, Config.IMG_HEIGHT))
        img = (img.astype(np.float32) / 255 - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
        img = torch.from_numpy(img).permute(2, 0, 1).float()
        
        enc = torch.tensor(
            [char_to_idx[c] for c in self.labels[idx] if c in char_to_idx],
            dtype=torch.long
        )
        
        return img, enc, len(enc)

def collate_fn(batch):
    imgs, lbls, lens = zip(*batch)
    imgs = torch.stack(imgs)
    max_len = max(lens)
    padded = torch.zeros(len(lbls), max_len, dtype=torch.long)
    for i, l in enumerate(lbls):
        padded[i, :len(l)] = l
    return imgs, padded, torch.tensor(lens, dtype=torch.long)

print("Dataset class defined!")

In [ ]:
# Model architecture - same as v3 (ResNet34)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(1))
    
    def forward(self, x):
        return self.dropout(x + self.pe[:x.size(0)])

class SVTR_v3(nn.Module):
    def __init__(self, vocab_size, d_model=384, nhead=8, num_layers=6, dim_ff=1024, dropout=0.15):
        super().__init__()
        
        # ResNet34 backbone
        resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        
        # Modify strides to preserve width
        resnet.layer2[0].conv1.stride = (2, 1)
        resnet.layer2[0].downsample[0].stride = (2, 1)
        resnet.layer3[0].conv1.stride = (2, 1)
        resnet.layer3[0].downsample[0].stride = (2, 1)
        resnet.layer4[0].conv1.stride = (2, 1)
        resnet.layer4[0].downsample[0].stride = (2, 1)
        
        self.backbone = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4
        )
        
        self.pool = nn.AdaptiveAvgPool2d((1, None))
        self.proj = nn.Conv2d(512, d_model, 1)
        self.pos = PositionalEncoding(d_model, dropout=dropout)
        
        enc_layer = nn.TransformerEncoderLayer(
            d_model, nhead, dim_ff, dropout, batch_first=False
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers)
        self.classifier = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.backbone(x)
        x = self.pool(x)
        x = self.proj(x).squeeze(2).permute(2, 0, 1)  # (W, B, C)
        x = self.pos(x)
        x = self.transformer(x)
        x = self.classifier(x)
        return F.log_softmax(x, dim=2)

print("Model architecture defined!")

In [ ]:
# Load data from unified_ocr_dataset
def load_data():
    # Real data
    df_real = pd.read_csv(Config.REAL_CSV)
    real_dir = os.path.join(Config.IMAGES_DIR, 'real')
    real_paths = [os.path.join(real_dir, r['id']) for _, r in df_real.iterrows()]
    real_labels = [str(r['text']) for _, r in df_real.iterrows()]
    
    # Filter existing files
    real_data = [(p, l) for p, l in zip(real_paths, real_labels) if os.path.exists(p)]
    real_paths, real_labels = zip(*real_data) if real_data else ([], [])
    
    # Synthetic data - زيادة طفيفة من 20K إلى 30K
    df_synth = pd.read_csv(Config.SYNTHETIC_CSV)
    synth_dir = os.path.join(Config.IMAGES_DIR, 'synthetic')
    synth_paths = [os.path.join(synth_dir, r['id']) for _, r in df_synth.iterrows()]
    synth_labels = [str(r['text']) for _, r in df_synth.iterrows()]
    
    synth_data = [(p, l) for p, l in zip(synth_paths, synth_labels) if os.path.exists(p)][:30000]
    synth_paths, synth_labels = zip(*synth_data) if synth_data else ([], [])
    
    return list(real_paths), list(real_labels), list(synth_paths), list(synth_labels)

real_p, real_l, synth_p, synth_l = load_data()
print(f"Real images: {len(real_p)}")
print(f"Synthetic images: {len(synth_p)}")

# Split real data
train_p, val_p, train_l, val_l = train_test_split(
    real_p, real_l, test_size=0.2, random_state=42
)

# Create datasets
synth_ds = OCRDataset(synth_p, synth_l, augment=False)
real_ds = OCRDataset(train_p, train_l, augment=True)
val_ds = OCRDataset(val_p, val_l, augment=False)

# Mix with 30x real data replication (same as v3)
mixed_ds = ConcatDataset([synth_ds] + [real_ds] * 30)

print(f"Training set size: {len(mixed_ds)} samples")
print(f"Validation set size: {len(val_ds)} samples")

# Data loaders (num_workers=0 for Jupyter compatibility)
train_loader = DataLoader(
    mixed_ds,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

print("Data loaders ready!")

In [ ]:
# Training and validation functions
def greedy_decode(preds):
    """CTC greedy decoder"""
    _, idx = preds.max(2)
    result = []
    for i in range(idx.size(1)):
        chars, prev = [], -1
        for j in idx[:, i].cpu().numpy():
            if j != 0 and j != prev and j in idx_to_char:
                chars.append(idx_to_char[j])
            prev = j
        result.append(''.join(chars))
    return result

def cer(preds, targets):
    """Character Error Rate"""
    errs, chars = 0, 0
    for p, t in zip(preds, targets):
        m, n = len(p), len(t)
        dp = [[0]*(n+1) for _ in range(m+1)]
        for i in range(m+1): dp[i][0] = i
        for j in range(n+1): dp[0][j] = j
        for i in range(1, m+1):
            for j in range(1, n+1):
                dp[i][j] = dp[i-1][j-1] if p[i-1]==t[j-1] else 1+min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
        errs += dp[m][n]
        chars += max(n, 1)
    return errs/chars if chars else 0

def train_epoch(model, loader, opt, crit, device):
    model.train()
    loss_sum = 0
    
    pbar = tqdm(loader, desc="Train", leave=False)
    for imgs, lbls, lens in pbar:
        imgs, lbls = imgs.to(device), lbls.to(device)
        opt.zero_grad()
        
        out = model(imgs)
        loss = crit(
            out, lbls,
            torch.full((imgs.size(0),), out.size(0), dtype=torch.long),
            lens
        )
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        opt.step()
        
        loss_sum += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return loss_sum / len(loader)

def validate(model, loader, crit, device):
    model.eval()
    loss_sum, preds, tgts = 0, [], []
    
    pbar = tqdm(loader, desc="Val", leave=False)
    with torch.no_grad():
        for imgs, lbls, lens in pbar:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out = model(imgs)
            
            loss_sum += crit(
                out, lbls,
                torch.full((imgs.size(0),), out.size(0), dtype=torch.long),
                lens
            ).item()
            
            preds.extend(greedy_decode(out))
            
            for i in range(lbls.size(0)):
                tgts.append(''.join([idx_to_char.get(j.item(),'') for j in lbls[i,:lens[i]]]))
    
    return loss_sum/len(loader), cer(preds, tgts)

print("Training functions defined!")

In [ ]:
# Initialize model and training components
model = SVTR_v3(
    NUM_CLASSES,
    Config.D_MODEL,
    Config.NHEAD,
    Config.NUM_LAYERS,
    Config.DIM_FF,
    Config.DROPOUT
).to(Config.DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

# Loss and optimizer (same as v3)
criterion = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=Config.LR,
    weight_decay=0.01
)

# ReduceLROnPlateau scheduler (same as v3)
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
    verbose=True
)

print("\nStarting training...")
print("="*60)

In [ ]:
# Main training loop
best_cer = float('inf')
no_improve = 0
history = {'train_loss': [], 'val_loss': [], 'val_cer': [], 'lr': []}

for epoch in range(Config.EPOCHS):
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, criterion, Config.DEVICE)
    
    # Validate
    val_loss, val_cer = validate(model, val_loader, criterion, Config.DEVICE)
    
    # Update scheduler
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_cer'].append(val_cer)
    history['lr'].append(current_lr)
    
    # Print progress
    if val_cer < best_cer:
        best_cer = val_cer
        no_improve = 0
        
        # Save best model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'cer': val_cer,
            'vocab': (char_to_idx, idx_to_char, NUM_CLASSES)
        }, os.path.join(Config.BASE_DIR, "svtr_v3_unified_best.pth"))
        
        print(f"★ Epoch {epoch+1}/{Config.EPOCHS} | Loss: {train_loss:.4f} | CER: {val_cer:.4f} | LR: {current_lr:.2e} ★")
    else:
        no_improve += 1
        print(f"  Epoch {epoch+1}/{Config.EPOCHS} | Loss: {train_loss:.4f} | CER: {val_cer:.4f} | LR: {current_lr:.2e}")
    
    # Early stopping
    if no_improve >= Config.PATIENCE:
        print(f"\n⚠ Early stopping at epoch {epoch+1}!")
        break

print(f"\n{'='*60}")
print(f"✓ Training completed!")
print(f"✓ Best CER: {best_cer:.4f} ({best_cer*100:.2f}%)")
print(f"{'='*60}")

In [ ]:
# Load best model and test
checkpoint = torch.load(os.path.join(Config.BASE_DIR, "svtr_v3_unified_best.pth"))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"\nBest model from epoch {checkpoint['epoch']+1}")
print(f"Best CER: {checkpoint['cer']:.4f} ({checkpoint['cer']*100:.2f}%)")

# Test on validation set
print("\nSample predictions:")
print("="*60)

with torch.no_grad():
    for i in range(min(10, len(val_p))):
        img = cv2.imread(val_p[i])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (Config.IMG_WIDTH, Config.IMG_HEIGHT))
        img = (img.astype(np.float32)/255 - [0.485,0.456,0.406]) / [0.229,0.224,0.225]
        img = torch.from_numpy(img).permute(2,0,1).float().unsqueeze(0).to(Config.DEVICE)
        
        out = model(img)
        pred = greedy_decode(out)[0]
        
        match = "✓" if pred == val_l[i] else "✗"
        print(f"{match} True: {val_l[i]}")
        print(f"  Pred: {pred}")
        print()

## Summary

### Architecture (same as v3):
- **Backbone**: ResNet34
- **Transformer**: d=384, 6 layers, 8 heads
- **Scheduler**: ReduceLROnPlateau
- **LR**: 3e-4

### Small improvements:
1. ✅ Training on unified_ocr_dataset
2. ✅ Increased synthetic data: 20K → 30K
3. ✅ Increased epochs: 50 → 60
4. ✅ Fixed num_workers for Jupyter

### Expected Results:
- Target CER: < 6.0% (similar to v3's 7.02%)
- Training time: ~1-2 hours on GPU
- Model saved: `svtr_v3_unified_best.pth`